In [1]:
%load_ext autoreload
%autoreload 2

import carla
from autonomy.config import MODEL_PATH
from autonomy.carla_env import connect, hide_map_layers, spawn_vehicle, CameraRig, SpeedController
from autonomy.model import SteeringModel, get_device
from autonomy.preprocessing import crop_to_rgb

In [2]:
client, world = connect()
print(world)

World(id=5413634456309708926)


In [3]:
# same road-only map as the data collection run, so the camera images match the training data
hide_map_layers(world)

Hid 232 environment objects


In [4]:
# spawn the car at the same fixed spawn point as data collection
# re-running would collide with the previous car at the same spot, so remove it first
try:
    vehicle.destroy()
except NameError:
    pass

vehicle = spawn_vehicle(world)

Spawned vehicle.tesla.model3 (id 24) at spawn point 5: Location(x=-106.649590, y=-17.073978, z=0.600000)


In [5]:
# forward facing camera attached to the ego vehicle
# re-running would leave the old camera attached, so remove it first
try:
    rig.destroy()
except NameError:
    pass

rig = CameraRig(world, vehicle)

Spawned camera (id 25) attached to vehicle 24


In [6]:
# load the trained steering model
device = get_device()
model = SteeringModel.load(MODEL_PATH, device)
print(f"Loaded {MODEL_PATH} on {device}")

Loaded steering_model.pt on cuda


In [7]:
# drive with the trained model: the network picks the steering, the speed controller picks the throttle.
# interrupt the kernel to stop; the car then brakes.
speed_controller = SpeedController(vehicle)
rig.clear()  # drop stale frames from before the run

try:
    while True:
        world.wait_for_tick()  # only update once per simulator tick

        steer = model.predict(crop_to_rgb(rig.latest()))
        throttle = speed_controller.step()
        vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=steer))
        print(f"speed {speed_controller.speed:.2f}  steer {steer:+.3f}")
except KeyboardInterrupt:
    pass
finally:
    vehicle.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0))
    print("Stopped")

speed 0.22  steer +0.024
speed 0.21  steer +0.024
speed 0.21  steer +0.024
speed 0.20  steer +0.024
speed 0.20  steer +0.024
speed 0.20  steer +0.024
speed 0.19  steer +0.024
speed 0.19  steer +0.024
speed 0.18  steer +0.024
speed 0.18  steer +0.024
speed 0.17  steer +0.024
speed 0.17  steer +0.024
speed 0.17  steer +0.024
speed 0.16  steer +0.024
speed 0.16  steer +0.024
speed 0.15  steer +0.024
speed 0.15  steer +0.024
speed 0.14  steer +0.024
speed 0.14  steer +0.024
speed 0.13  steer +0.024
speed 0.13  steer +0.024
speed 0.12  steer +0.024
speed 0.12  steer +0.024
speed 0.12  steer +0.024
speed 0.11  steer +0.024
speed 0.11  steer +0.024
speed 0.11  steer +0.024
speed 0.10  steer +0.024
speed 0.10  steer +0.024
speed 0.10  steer +0.024
speed 0.09  steer +0.024
speed 0.09  steer +0.024
speed 0.09  steer +0.024
speed 0.08  steer +0.024
speed 0.08  steer +0.024
speed 0.08  steer +0.024
speed 0.08  steer +0.024
speed 0.07  steer +0.024
speed 0.07  steer +0.024
speed 0.07  steer +0.024
